## 🎯 Learning Objectives
* Understand the limitations of standard RAG for complex, multi-hop queries.
* Grasp the concept of recursive retrieval for multi-stage information gathering.
* Learn about small-to-big retrieval for balancing detail and context in RAG.
* Implement recursive and small-to-big retrieval patterns using LlamaIndex.


## Advanced RAG Patterns: Recursive and Small-to-Big Retrieval

In the rapidly evolving landscape of Agentic AI, Retrieval Augmented Generation (RAG) has become a cornerstone for building intelligent systems that can ground their responses in factual, external knowledge. However, as queries become more complex and user expectations rise, standard RAG approaches often hit limitations. This lesson explores two advanced patterns – **Recursive Retrieval** and **Small-to-Big Retrieval** – designed to overcome these challenges and deliver more robust, accurate, and contextually rich answers.

### The Need for Advanced Retrieval

Traditional RAG typically involves a single-pass retrieval: a user query is embedded, similar documents are retrieved, and these documents are passed to a Large Language Model (LLM) for synthesis. This works well for straightforward questions, but struggles with:

1.  **Multi-hop Questions:** Queries that require synthesizing information from multiple, disparate sources or require intermediate steps of reasoning. For example, "What is the capital of the country where the inventor of Python was born?"
2.  **Context Granularity Issues:** When documents are chunked too small, the LLM might lack sufficient context to answer comprehensively. If chunks are too large, the LLM might suffer from the "lost in the middle" problem, where crucial information is overlooked within a vast context window.

### Recursive Retrieval: The Detective's Approach

Imagine a detective solving a complex case. They don't just look at one piece of evidence and declare the case closed. Instead, an initial clue leads them to a witness, who provides new information, which in turn points to a different location, and so on. This iterative process of gathering information, analyzing it, and generating new leads continues until the full picture emerges. This is the essence of **Recursive Retrieval**.

In RAG, recursive retrieval means that the LLM doesn't just answer the initial query. Instead, it can analyze the initial retrieval results, identify follow-up questions, entities, or missing information, and then generate *new queries* to retrieve more relevant data. This process can repeat multiple times, allowing the system to progressively build a more complete understanding before formulating a final answer.

**How it works (Step-by-Step):**
1.  **Initial Query:** The user asks a complex question.
2.  **First Retrieval:** The system performs an initial retrieval based on the user's query.
3.  **LLM Analysis & Sub-Question Generation:** An LLM reviews the initial retrieved documents. If it determines that the initial information is insufficient or that the query requires multiple steps, it generates one or more *sub-questions* or identifies key entities for further investigation.
4.  **Sub-Query Execution:** Each sub-question is treated as a new query, and the retrieval process (steps 2-3) is repeated for these sub-questions.
5.  **Synthesis:** Once all sub-questions are answered (or a predefined depth/iteration limit is reached), the LLM synthesizes all the gathered information to formulate a comprehensive final answer to the original user query.

### Small-to-Big Retrieval: Contextual Zoom-in

Consider how you might research a topic in a large textbook. You wouldn't read the entire book to find one fact. Instead, you might start by scanning the table of contents or chapter summaries (small chunks of information) to identify relevant sections. Once you find a promising section, you'd then dive into the detailed paragraphs within that section (larger chunks of information) to extract the specific details you need. This is **Small-to-Big Retrieval**.

This pattern addresses the context granularity problem by using different-sized chunks for different stages of the RAG process:

1.  **Small Chunks for Retrieval:** Smaller, more granular chunks (e.g., individual sentences, short paragraphs) are used for embedding and similarity search. These smaller chunks are excellent for precise semantic matching and reducing the chance of irrelevant information diluting the embedding.
2.  **Big Chunks for Context:** Once relevant small chunks are identified, their corresponding *larger parent chunks* (e.g., full paragraphs, sections, or even entire documents) are retrieved and passed to the LLM. These larger chunks provide the necessary surrounding context for the LLM to understand the retrieved information fully and synthesize a coherent answer, mitigating the "lost in the middle" problem without sacrificing retrieval precision.

**How it works (Step-by-Step):**
1.  **Dual Indexing:** The original documents are processed into two sets of chunks:
    *   **Small Chunks (Child Nodes):** Used for vector embedding and similarity search.
    *   **Big Chunks (Parent Nodes):** The larger, contextual blocks that contain the small chunks. These are stored separately, often in a document store, and linked to their child nodes.
2.  **Query & Small Chunk Retrieval:** The user's query is embedded, and a similarity search is performed against the *vector store of small chunks*.
3.  **Parent Chunk Expansion:** For each top-k small chunk retrieved, its corresponding *parent chunk* is identified and retrieved from the document store.
4.  **LLM Synthesis:** The collection of larger parent chunks is then passed to the LLM as context to generate the final answer.

Both recursive and small-to-big retrieval patterns represent significant advancements in building more intelligent and robust RAG systems, crucial for the complex demands of 2026 and beyond.


In [ ]:
# Install necessary libraries (as of 2026, LlamaIndex is a mature framework)
# !pip install llama-index openai pypdf

import os
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.retrievers import ParentDocumentRetriever
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.storage.index_store import SimpleIndexStore
from llama_index.core.vector_stores import SimpleVectorStore
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Set up OpenAI API Key (ensure it's in your environment variables)
# For 2026, assume robust environment variable management or secure key vault integration
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY" # Replace with your actual key or ensure env var is set

# Configure LlamaIndex global settings for LLM and Embedding Model
# As of 2026, gpt-4o and text-embedding-3-large are standard for high-performance RAG
Settings.llm = OpenAI(model="gpt-4o", temperature=0.1)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")

# --- 1. Data Preparation --- 
# Let's create some synthetic documents for demonstration
# In a real-world scenario, these would come from PDFs, databases, web pages, etc.
documents = [
    Document(text="""
    AgenticLabs.ng is a leading AI research and development firm based in Lagos, Nigeria.
    Founded in 2022 by Dr. Adaeze Okoro, a renowned expert in reinforcement learning,
    the company quickly established itself as a pioneer in developing autonomous AI agents.
    Their flagship product, 'Aether', is an advanced automation platform for enterprise operations.
    """),
    Document(text="""
    Dr. Adaeze Okoro completed her Ph.D. in Computer Science at Stanford University in 2018.
    Her doctoral research focused on novel algorithms for multi-agent systems and distributed AI.
    Before founding AgenticLabs.ng, she worked as a senior AI scientist at DeepMind in London.
    She is also an avid marathon runner and a strong advocate for STEM education in Africa.
    """),
    Document(text="""
    The Aether platform by AgenticLabs.ng leverages a modular architecture, integrating
    advanced LLMs with specialized tools for data analysis, process automation, and decision support.
    It is primarily deployed in the financial services and logistics sectors, where it optimizes
    complex workflows and reduces operational costs by up to 30%. The platform supports
    customizable agent personas and real-time monitoring dashboards.
    """)
]

print("--- Documents Loaded ---")
for i, doc in enumerate(documents):
    print(f"Document {i+1}: {doc.text[:100]}...")

# --- 2. Recursive Retrieval Example (using SubQuestionQueryEngine) ---
print("\n--- Demonstrating Recursive Retrieval ---")

# Create a standard VectorStoreIndex from our documents
vector_index = VectorStoreIndex.from_documents(documents)

# Create a QueryEngineTool for our index
# The SubQuestionQueryEngine needs to know which query engines it can use
query_engine_tool = QueryEngineTool(
    query_engine=vector_index.as_query_engine(),
    metadata=ToolMetadata(
        name="agentic_labs_info",
        description="Provides information about AgenticLabs.ng, its founder, and products."
    ),
)

# Initialize the SubQuestionQueryEngine
# This engine will break down complex questions into simpler sub-questions
# and use the provided tools to answer them iteratively.
sub_question_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=[query_engine_tool],
    llm=Settings.llm, # Use the globally configured LLM
    verbose=True # Set to True to see the sub-questions generated
)

# Example of a multi-hop query
recursive_query = "What is the main product of the company founded by the person who worked at DeepMind, and what sectors does it serve?"
print(f"\nOriginal Query: {recursive_query}")
recursive_response = sub_question_engine.query(recursive_query)
print(f"\nRecursive Retrieval Final Answer: {recursive_response}")

# --- 3. Small-to-Big Retrieval Example (using ParentDocumentRetriever) ---
print("\n--- Demonstrating Small-to-Big Retrieval ---")

# Configure chunk sizes
# Small chunks for embedding and retrieval (e.g., sentences)
# Big chunks for LLM context (e.g., paragraphs or larger sections)
child_splitter = SentenceSplitter(chunk_size=128, chunk_overlap=20)
# Parent splitter can be larger, or even the original document if desired
parent_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)

# Initialize storage contexts for parent and child nodes
# Simple stores are used here; in production, use persistent stores like Chroma, Pinecone, etc.
doc_store = SimpleDocumentStore()
vector_store = SimpleVectorStore()
index_store = SimpleIndexStore()

# Create the ParentDocumentRetriever
# It will store parent documents in doc_store and child chunks in vector_store
parent_document_retriever = ParentDocumentRetriever(
    vector_store=vector_store,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    # The original documents are passed here to be split into parent and child nodes
    # and stored in their respective stores.
    # This step effectively builds the dual-granularity index.
    # Note: For large datasets, this indexing can be time-consuming.
)

# Add documents to the retriever, which handles the splitting and storage
parent_document_retriever.add_documents(documents, show_progress=True)

# Create a query engine that uses the ParentDocumentRetriever
# The retriever will fetch small chunks, then expand to their parent chunks for the LLM.
parent_document_query_engine = VectorStoreIndex(
    nodes=parent_document_retriever.get_all_nodes(show_progress=True),
    vector_store=vector_store, # This is the vector store for child nodes
    docstore=doc_store, # This is the document store for parent nodes
).as_query_engine(retriever=parent_document_retriever)

# Example query for small-to-big retrieval
small_to_big_query = "Tell me about Dr. Adaeze Okoro's background and her contributions to AgenticLabs.ng."
print(f"\nOriginal Query: {small_to_big_query}")
small_to_big_response = parent_document_query_engine.query(small_to_big_query)
print(f"\nSmall-to-Big Retrieval Final Answer: {small_to_big_response}")

print("\n--- End of Demonstration ---")


### Interpreting the Output and Performance Considerations

#### Recursive Retrieval Output

When running the recursive retrieval example, you'll notice `SubQuestionQueryEngine`'s `verbose=True` setting provides valuable insights. It explicitly shows:

*   **Generated Sub-questions:** The LLM's breakdown of the original complex query into simpler, actionable sub-questions. For our example, "What is the main product of the company founded by the person who worked at DeepMind, and what sectors does it serve?" might be broken down into:
    1.  "Who worked at DeepMind?"
    2.  "What company did that person found?"
    3.  "What is the main product of that company?"
    4.  "What sectors does that product serve?"
*   **Intermediate Answers:** The responses obtained for each sub-question from the underlying `QueryEngineTool`.
*   **Final Synthesis:** The LLM's ultimate answer, combining the intermediate results to address the original query.

This transparency is crucial for debugging and understanding the reasoning path of the RAG system.

#### Small-to-Big Retrieval Output

For the small-to-big retrieval, the output will appear as a direct answer to your query. However, behind the scenes, the `ParentDocumentRetriever` performed a two-stage process:

1.  It first retrieved small, precise chunks (e.g., sentences) that semantically matched your query.
2.  Then, it expanded these small chunks to their larger parent documents (e.g., full paragraphs or sections) before passing them to the LLM. This ensures the LLM receives sufficient context, even if the initial match was on a very specific phrase.

#### Performance Trade-offs

Both advanced patterns offer significant benefits but come with their own performance considerations:

**Recursive Retrieval:**
*   **Pros:** Highly effective for multi-hop reasoning, complex analytical queries, and situations where the optimal information path isn't clear upfront. Reduces the risk of "missing context" by actively seeking out necessary information.
*   **Cons:** Higher latency and increased computational cost due to multiple LLM calls and retrieval steps. Each sub-question incurs an LLM inference and potentially a vector database lookup. Requires careful prompt engineering for sub-question generation to avoid infinite loops or irrelevant tangents.

**Small-to-Big Retrieval:**
*   **Pros:** Balances retrieval precision (using small chunks) with contextual richness (using large chunks). Effectively mitigates the "lost in the middle" problem by providing the LLM with relevant, yet sufficiently broad, context. Improves answer quality for detailed questions.
*   **Cons:** Increased indexing complexity and storage requirements. You need to manage two granularities of chunks and their mapping. The initial indexing process can be more time-consuming. Retrieval latency might be slightly higher than single-pass RAG due to the additional step of fetching parent documents.

#### Typical Use Cases (2026 Perspective)

As LLMs become more powerful and context windows expand, these patterns are becoming standard for enterprise-grade RAG applications:

*   **Recursive Retrieval:**
    *   **Complex Q&A Systems:** Answering intricate questions from large knowledge bases (e.g., legal documents, scientific papers, internal company wikis) that require synthesizing information across multiple sections or documents.
    *   **Automated Research Agents:** Agents that can autonomously explore a topic, identify gaps in knowledge, and iteratively gather information.
    *   **Knowledge Graph Construction:** Identifying entities and relationships by iteratively querying and refining information.

*   **Small-to-Big Retrieval:**
    *   **Detailed Document Analysis:** Extracting specific facts while maintaining the surrounding context for nuanced understanding (e.g., financial reports, medical records).
    *   **Long-Form Content Generation:** Ensuring generated content is grounded in precise details but also flows coherently with broader context.
    *   **Legal and Compliance RAG:** Where specific clauses or definitions need to be retrieved, but the full legal context of the paragraph or section is critical for interpretation.

These patterns are essential tools in the modern RAG engineer's toolkit, enabling the construction of highly capable and reliable AI agents.


### Resources

*   **LlamaIndex Documentation - SubQuestionQueryEngine:** [https://docs.llamaindex.ai/en/stable/module_guides/querying/query_engines/sub_question_query_engine.html](https://docs.llamaindex.ai/en/stable/module_guides/querying/query_engines/sub_question_query_engine.html)
*   **LlamaIndex Documentation - ParentDocumentRetriever:** [https://docs.llamaindex.ai/en/stable/module_guides/retrievers/parent_document_retriever.html](https://docs.llamaindex.ai/en/stable/module_guides/retrievers/parent_document_retriever.html)
*   **LlamaIndex Documentation - Node Parsers (Chunking Strategies):** [https://docs.llamaindex.ai/en/stable/module_guides/loading/node_parsers/root.html](https://docs.llamaindex.ai/en/stable/module_guides/loading/node_parsers/root.html)
*   **OpenAI API Documentation:** [https://platform.openai.com/docs/overview](https://platform.openai.com/docs/overview)
*   **Research Paper - "Lost in the Middle: How Language Models Can Lose Information When Context is Too Long" (Relevant for Small-to-Big RAG motivation):** [https://arxiv.org/abs/2307.03172](https://arxiv.org/abs/2307.03172)
